In [1]:
import os, sys
from vertex_algebras.universal_vertex_algebra import UniversalVertexAlgebra
from vertex_algebras.normally_ordered_expressions import NormallyOrderedExpressions
from vertex_algebras.vertex_algebra import VertexAlgebraParallelism

In [2]:
# Parameters: change these freely, then rerun the notebook
cartan = ["A", 2]      # any simple type, e.g. ["B", 2], ["C", 3], ["G", 2]
sv_weight = 2          # conformal weight of the singular vector
sv_grade = None        # root lattice grade: None = highest root, a root lattice element, or a tuple in simple root coordinates

In [3]:
# Construct a simple Lie algebra of any type over the rational field QQ in the Chevalley basis
g = LieAlgebra(QQ, cartan_type=cartan)
basis = g.basis()
bkeys = list(basis.keys())   # to_vector() coordinates are in exactly this order
labels = [str(k) for k in bkeys]   # string generator labels, e.g. 'alpha[1]', 'alphacheck[2]'

In [4]:
rg = g.cartan_type().root_system().root_lattice() # Find the basis root vector corresponding to the highest root to normalise the Killing form such that the highest root has norm square 2
highest_root = rg.highest_root()
e_highest = basis[highest_root]
current_norm_sq = e_highest.killing_form(basis[-highest_root])
scaling_factor = 1 / current_norm_sq
def normalised_killing_form(x, y):
    return scaling_factor * x.killing_form(y)

In [5]:
# Affine vertex algebra: OPE(x, y) = [x,y]/(z-w) + k*B(x,y)/(z-w)^2
R.<k> = Frac(QQ['k'])   # fractions in k: the Sugawara vector below divides by k + h^vee
E = NormallyOrderedExpressions(R, labels)   # the currents, to write the OPEs before the algebra exists

def to_va(v):
    """Expand a Lie algebra element in the Chevalley basis as a VA element."""
    vec = v.to_vector()
    return sum((c * E.gen(lab) for c, lab in zip(vec, labels) if c), E.zero()) # c is the coefficient of the basis label lab. If c excludes all basis vectors with coefficient 0.

opes = {}
for i, a in enumerate(bkeys):
    for j in range(i, len(bkeys)):   # i <= j: includes diagonal (x, x)
        b = bkeys[j]
        x, y = basis[a], basis[b]
        opes[(labels[i], labels[j])] = [
            to_va(x.bracket(y)),
            k * normalised_killing_form(x, y),
        ]
VertexAlgebraParallelism().set(20)   # threads of the C++ engine, for the whole session
affVA = UniversalVertexAlgebra(R, E, opes)

In [6]:
# Sanity checks
I = g.cartan_type().index_set()
cor = g.cartan_type().root_system().coroot_lattice()
hs = [affVA.gen(str(cor.simple_root(i))) for i in I]      # the simple coroots alphacheck[i]
print('Cartan (abelian):', all(x.ope(y)[0] == 0 for x in hs for y in hs if x.ope(y)))
print('Cartan second poles / k (normalised form on simple coroots):')
print(matrix(QQ, [[normalised_killing_form(basis[cor.simple_root(i)], basis[cor.simple_root(j)]) for j in I] for i in I]))
print('h1,h1 :', hs[0].ope(hs[0]))

lab_th = next(lab for lab, key in zip(labels, bkeys) if key == highest_root)
lab_nth = next(lab for lab, key in zip(labels, bkeys) if key == -highest_root)
e_th = affVA.gen(lab_th)
e_nth = affVA.gen(lab_nth)
ophn = e_th.ope(e_nth)
theta_vee = highest_root.associated_coroot()                    # expected first pole: theta^vee in the simple coroots
print('[e_theta, e_-theta] first pole:', ophn[0], ' = theta^vee:',
      ophn[0] == sum(c * affVA.gen(str(cor.simple_root(i))) for i, c in theta_vee))
print('[e_theta, e_-theta] second pole:', ophn[1])            # expect k: the highest root has norm square 2

op_rev = e_nth.ope(e_th)
print('skew first pole ok :', op_rev[0] == -ophn[0])
print('skew second pole ok:', op_rev[1] == ophn[1])
affVA

Cartan (abelian): True
Cartan second poles / k (normalised form on simple coroots):
[ 2 -1]
[-1  2]
h1,h1 : [0, 2*k]
[e_theta, e_-theta] first pole: alphacheck[1] + alphacheck[2]  = theta^vee: True
[e_theta, e_-theta] second pole: k
skew first pole ok : True
skew second pole ok: True


Universal vertex algebra on generators ['alpha[2]', 'alpha[1]', 'alpha[1] + alpha[2]', 'alphacheck[1]', 'alphacheck[2]', '-alpha[2]', '-alpha[1]', '-alpha[1] - alpha[2]'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field

In [7]:
# Grading by the root lattice: e_alpha has grade alpha, the Cartan generators grade 0.
# Grades are coordinate vectors in the simple roots, in ZZ^rank.
def root_grade(key):
    if key in rg:                                  # a root
        return tuple(ZZ(rg(key).coefficient(i)) for i in I)
    return (0,) * len(I)                           # alphacheck[i]
affVA = affVA.with_horizontal_grading({lab: root_grade(key) for lab, key in zip(labels, bkeys)})
print(affVA.horizontal_grading_group())
{lab: affVA.gen(lab).horizontal_grade() for lab in labels}

Ambient free module of rank 2 over the principal ideal domain Integer Ring


{'alpha[2]': (0, 1),
 'alpha[1]': (1, 0),
 'alpha[1] + alpha[2]': (1, 1),
 'alphacheck[1]': (0, 0),
 'alphacheck[2]': (0, 0),
 '-alpha[2]': (0, -1),
 '-alpha[1]': (-1, 0),
 '-alpha[1] - alpha[2]': (-1, -1)}

In [8]:
# Sugawara vector  T = 1/(2(k + h^vee)) sum_i :J_i J^i:,  J^i the dual basis for the normalised Killing form
G = matrix(QQ, [[normalised_killing_form(basis[a], basis[b]) for b in bkeys] for a in bkeys])
Ginv = G.inverse()                                 # J^i = sum_j Ginv[i, j] J_j
J = [affVA.gen(lab) for lab in labels]
hvee = g.cartan_type().dual_coxeter_number()
T_sug = sum(Ginv[i, j] * J[i] * J[j] for i in range(len(J)) for j in range(len(J)) if Ginv[i, j]) / (2 * (k + hvee))
affVA = affVA.with_conformal_vector(T_sug)         # checks the Virasoro OPE and that T_(0) = ∂
T_sug = affVA.conformal_vector()
c_expected = k * g.dimension() / (k + hvee)
print('central charge:', affVA.central_charge(), ' expected:', c_expected, affVA.central_charge() == c_expected)
print('conformal weights:', affVA.degrees())
print('T grade and weight:', T_sug.horizontal_grade(), T_sug.degree())

central charge: 8*k/(k + 3)  expected: 8*k/(k + 3) True
conformal weights: {'alpha[2]': 1, 'alpha[1]': 1, 'alpha[1] + alpha[2]': 1, 'alphacheck[1]': 1, 'alphacheck[2]': 1, '-alpha[2]': 1, '-alpha[1]': 1, '-alpha[1] - alpha[2]': 1}
T grade and weight: (0, 0) 2


In [9]:
# Weight spaces now take both a conformal weight and a root lattice grade
alpha1 = root_grade(rg.simple_root(I[0]))          # grade of the first simple root
zero = root_grade(None)                            # grade 0
print(affVA.weight_space(1, horizontal_grade=alpha1).basis())
S = affVA.weight_space(2, horizontal_grade=zero)
print(S)                                           # dimension: #positive roots + rank(rank+1)/2 + rank
S.basis()

[alpha[1]]
Weight space of conformal weight 2 and grade (0, 0) of Universal vertex algebra on generators ['alpha[2]', 'alpha[1]', 'alpha[1] + alpha[2]', 'alphacheck[1]', 'alphacheck[2]', '-alpha[2]', '-alpha[1]', '-alpha[1] - alpha[2]'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field (dimension 8)


[:alpha[2] -alpha[2]:,
 :alpha[1] -alpha[1]:,
 :alpha[1] + alpha[2] -alpha[1] - alpha[2]:,
 :alphacheck[1] alphacheck[1]:,
 :alphacheck[1] alphacheck[2]:,
 dalphacheck[1],
 :alphacheck[2] alphacheck[2]:,
 dalphacheck[2]]

## Singular vectors of given conformal weight and grade

Method of `psl22_principal.ipynb`: write the general vector $v$ of the weight space as a coefficient vector, turn each condition
`a.nproduct(v, n - 1) = 0` into a matrix over $\mathbb{Q}[k]$, stack them into $A$ and read the levels where $\ker A \neq 0$ off the
elementary divisors of $A$ (Smith form over the PID $\mathbb{Q}[k]$).

The VA is defined over $\mathrm{Frac}(\mathbb{Q}[k])$ (for the Sugawara vector), but all current OPEs are polynomial in $k$, so the
matrix entries are converted to $\mathbb{Q}[k]$ (an assertion checks the denominators are 1).

$v$ is singular in the vacuum module when $\hat{\mathfrak g}_+ = \mathfrak n_+ \oplus \mathfrak g \otimes t\mathbb{C}[t]$ annihilates it.
As a Lie algebra $\hat{\mathfrak g}_+$ is generated by $(e_{\alpha_i})_0$ for the simple roots and $(e_{-\theta})_1$, so it suffices to impose

| field | pole | mode |
|---|---|---|
| $e_{\alpha_i}$, $i$ simple | order 1 | $(e_{\alpha_i})_{(0)} v = 0$ |
| $e_{-\theta}$ | order 2 | $(e_{-\theta})_{(1)} v = 0$ |

(`a.nproduct(v, n - 1)` $= a_{(n-1)} v$.) These are rank$+1$ conditions, independent of the type.

In [10]:
P = R.ring()                                       # QQ['k'], a PID: needed for elementary divisors
if sv_grade is None:
    sv_grade = highest_root
target_grade = tuple(sv_grade) if isinstance(sv_grade, (tuple, list)) else root_grade(rg(sv_grade))
SVspace = affVA.weight_space(sv_weight, horizontal_grade=target_grade)
print(SVspace)
SVspace.basis()

Weight space of conformal weight 2 and grade (1, 1) of Universal vertex algebra on generators ['alpha[2]', 'alpha[1]', 'alpha[1] + alpha[2]', 'alphacheck[1]', 'alphacheck[2]', '-alpha[2]', '-alpha[1]', '-alpha[1] - alpha[2]'] over Fraction Field of Univariate Polynomial Ring in k over Rational Field (dimension 4)


[:alpha[2] alpha[1]:,
 :alpha[1] + alpha[2] alphacheck[1]:,
 :alpha[1] + alpha[2] alphacheck[2]:,
 dalpha[1] + alpha[2]]

In [11]:
def pole_matrix(a, n, source):
    """
    The matrix over QQ[k] of  x |-> a.nproduct(x, n - 1)  from the weight space `source` to the weight space the images land in.
    Column j holds the coefficients of a.nproduct(b_j, n - 1) in the basis of the target space. Returns (matrix, target space).
    """
    W = a.degree() + source.weight() - n
    G = tuple(x + y for x, y in zip(a.horizontal_grade(), source.horizontal_grade()))
    target = affVA.weight_space(W, horizontal_grade=G)
    row = {m: r for r, m in enumerate(target.basis_keys())}
    basis = source.basis()
    M = matrix(P, len(row), len(basis))
    for j, b in enumerate(basis):
        for m, c in a.nproduct(b, n - 1).monomial_coefficients().items():
            assert c.denominator() == 1                # the current OPEs are polynomial in k
            M[row[m], j] = P(c.numerator())
    return M, target

def solution_dimensions(A):
    """
    (generic dimension of {a : A a = 0}, {k0: dimension at k = k0} for the k0 where it jumps).
    """
    n = A.ncols()
    nonzero = [d for d in A.elementary_divisors() if d != 0]
    special = {}
    for d in nonzero:
        if d.degree() > 0:
            for k0, _ in d.roots(QQbar):
                k0 = QQ(k0) if k0 in QQ else k0            # rational levels as elements of QQ, the others stay in QQbar
                special[k0] = n - sum(1 for e in nonzero if e(k0) != 0)
    return n - len(nonzero), special

def vectors_from(rows, source):
    """Coefficient vectors (w.r.t. source.basis()) -> elements of the VA."""
    return [sum(c * b for c, b in zip(row, source.basis())) for row in rows]

In [12]:
# The generators of g^_+ : (e_{alpha_i})_(0) for the simple roots and (e_{-theta})_(1)
conditions = [(affVA.gen(str(rg.simple_root(i))), 1) for i in I] + [(affVA.gen(str(-highest_root)), 2)]
print([(str(a), n) for a, n in conditions])

A = block_matrix([[pole_matrix(a, n, SVspace)[0]] for a, n in conditions], subdivide=False)
ed = A.elementary_divisors()
print('A is', A.dimensions(), ' elementary divisors:', ed)
nonconst = [d for d in ed if d != 0 and d.degree() > 0]
if nonconst:
    print('last elementary divisor factors as', nonconst[-1].factor())
solution_dimensions(A)

[('alpha[1]', 1), ('alpha[2]', 1), ('-alpha[1] - alpha[2]', 2)]
A is (4, 4)  elementary divisors: [1, 1, 1, 2*k + 3]
last elementary divisor factors as (2) * (k + 3/2)


(0, {-3/2: 1})

The singular vectors at the special levels, and a direct check of the conditions there (and, as a sanity check, of all positive modes of all currents):

In [13]:
def vanishes_at(x, k0):
    """True if every coefficient of x (over Frac(QQ[k])) vanishes at k = k0."""
    return all(c.numerator()(k0) == 0 for c in x.monomial_coefficients().values())

def all_positive_modes(v):
    """The conditions a.nproduct(v, n - 1) = 0 for the full g^_+: poles n >= 2 of every current, n = 1 of the positive roots."""
    out = []
    for lab, key in zip(labels, bkeys):
        a = affVA.gen(lab)
        nmin = 1 if (key in rg and rg(key).is_positive_root()) else 2
        out += [(a, n) for n in range(nmin, sv_weight + 2)]      # a_(m) v = 0 automatically for m > sv_weight
    return out

singular = {}
for k0 in solution_dimensions(A)[1]:
    F = QQ if k0 in QQ else QQbar
    A0 = A.apply_map(lambda p: p(k0), F)           # the matrix at k = k0
    if F is not QQ:                                # irrational level: the VA is over QQ(k), so show coefficient vectors
        print('k =', k0, ': kernel rows w.r.t. SVspace.basis():', A0.right_kernel_matrix().rows())
        continue
    singular[k0] = vectors_from(A0.right_kernel_matrix().rows(), SVspace)
    for v in singular[k0]:
        print('k =', k0, ':', v)
        print('   generating conditions hold:', all(vanishes_at(a.nproduct(v, n - 1), k0) for a, n in conditions),
              '  all of g^_+:', all(vanishes_at(a.nproduct(v, n - 1), k0) for a, n in all_positive_modes(v)))

k = -3/2 : :alpha[2] alpha[1]: - 1/3*:alpha[1] + alpha[2] alphacheck[1]: + 1/3*:alpha[1] + alpha[2] alphacheck[2]: - 1/2*dalpha[1] + alpha[2]
   generating conditions hold: True   all of g^_+: True


## Example: $D_4^{(1)}$ at level $k = -2$: three singular vectors and sums of ideals

At $k = -2$ the universal affine vertex algebra $V^{-2}(\mathfrak{so}_8)$ has three singular vectors of conformal weight 2, of
$\mathfrak{so}_8$-weights $2\omega_1$, $2\omega_3$, $2\omega_4$. Triality permutes them; none is a multiple of another.

- Perše, [arXiv:1205.3003](https://arxiv.org/abs/1205.3003), §4, writes them out and proves (Thm 4.2) that they generate the maximal ideal.
- Arakawa–Moreau, [arXiv:1506.00710](https://arxiv.org/abs/1506.00710), Thm 4.2, construct them uniformly for the Deligne series, $k = -h^\vee/6 - 1$.

So $\langle v_1\rangle$, $\langle v_3\rangle$, $\langle v_4\rangle$ are three distinct proper ideals whose sums are all different, which makes a good
example for sums of ideals.

This section does not depend on the parameters at the top. It builds its own $D_4$ algebras and reuses `R`, `P`, `solution_dimensions` and `vectors_from` from above.

In [14]:
# so(8) = D4 with the construction of the cells above, but for a given type, level and coefficient ring, so this section
# runs whatever the parameters at the top are.
def affine_vertex_algebra(cartan, level, ring):
    g = LieAlgebra(QQ, cartan_type=cartan)
    basis = g.basis()
    bkeys = list(basis.keys())
    labels = [str(b) for b in bkeys]
    rg = g.cartan_type().root_system().root_lattice()
    theta = rg.highest_root()
    scale = 1 / basis[theta].killing_form(basis[-theta])   # highest root of norm square 2
    E = NormallyOrderedExpressions(ring, labels)
    def to_va(x):
        return sum((c * E.gen(lab) for c, lab in zip(x.to_vector(), labels) if c), E.zero())
    index = g.cartan_type().index_set()
    V = UniversalVertexAlgebra(ring, E,
                               {(labels[i], labels[j]): [to_va(basis[a].bracket(basis[bkeys[j]])),
                                                         level * scale * basis[a].killing_form(basis[bkeys[j]])]
                                for i, a in enumerate(bkeys) for j in range(i, len(bkeys))},
                               horizontal_grading={lab: tuple(ZZ(rg(b).coefficient(i)) for i in index) if b in rg else (0,) * len(index)
                                        for lab, b in zip(labels, bkeys)},
                               degrees={lab: 1 for lab in labels})   # the Sugawara weights, without building T
    return V, rg

cartan_D4 = ['D', 4]
CI = CartanType(cartan_D4).cartan_matrix().inverse()
def two_omega(i):
    """2 * (fundamental weight i) in simple root coordinates (a root lattice grade)."""
    return tuple(ZZ(2 * c) for c in CI.row(i - 1))

def condition_matrix(V, rg, source, convert):
    """As pole_matrix above, stacked over the generators of g^_+ (for the algebra V): the conditions
    (e_{alpha_i})_(0) v = 0 and (e_{-theta})_(1) v = 0 on v in `source`; entries converted by `convert`."""
    conds = [(V.gen(str(a)), 1) for a in rg.simple_roots()] + [(V.gen(str(-rg.highest_root())), 2)]
    B = source.basis()
    blocks = []
    for a, n in conds:
        images = [a.nproduct(b, n - 1).monomial_coefficients() for b in B]
        keys = sorted({m for x in images for m in x})
        blocks.append(matrix([[convert(x.get(m, 0)) for x in images] for m in keys]))
    return block_matrix([[M] for M in blocks if M.nrows()], subdivide=False)

First the level. As in the section above, the singular vectors of weight 2 and grade $2\omega_i$ are the kernel of the
conditions over $\mathbb{Q}[k]$, and the elementary divisors show where a kernel appears.

In [15]:
# Generic level: the singular vectors of weight 2 in the grades 2ω1, 2ω3, 2ω4 exist only at k = -2
D4k, rg4 = affine_vertex_algebra(cartan_D4, k, R)
def to_P(c):
    assert c.denominator() == 1                    # the current OPEs are polynomial in k
    return P(c.numerator())
for i in (1, 3, 4):
    src = D4k.weight_space(2, horizontal_grade=two_omega(i))
    A4 = condition_matrix(D4k, rg4, src, to_P)
    print('2ω%d = %s:' % (i, two_omega(i)), 'weight space of dimension', src.dimension(),
          ' elementary divisors', A4.elementary_divisors(), ' kernel dimensions', solution_dimensions(A4))

2ω1 = (2, 2, 1, 1): weight space of dimension 3  elementary divisors [1, 1, -k - 2]  kernel dimensions (0, {-2: 1})


2ω3 = (1, 2, 2, 1): weight space of dimension 3  elementary divisors [1, 1, k + 2]  kernel dimensions (0, {-2: 1})


2ω4 = (1, 2, 1, 2): weight space of dimension 3  elementary divisors [1, 1, k + 2]  kernel dimensions (0, {-2: 1})


Ideals need the level fixed (over $\mathbb{Q}(k)$ these vectors are not singular), so the rest works over $\mathbb{Q}$ at $k = -2$.

In $\varepsilon$-coordinates, $v_3$ and $v_4$ are exactly Perše's $\theta(v)$ and $\theta^2(v)$. $v_1$ is his $v$ with the sign of one term
changed: the signs depend on the choice of root vectors (Sage's Chevalley basis against Bourbaki / Feingold–Frenkel).

In [16]:
# The algebra at k = -2 over QQ and its three singular vectors
D4, rg4 = affine_vertex_algebra(cartan_D4, -2, QQ)
gens4 = D4.gens()

def eps(root):
    """A root of D4 in the coordinates ε1, ..., ε4."""
    return ''.join('%s%s' % ('+' if c > 0 else '-', 'ε%d' % (j + 1))
                   for j, c in enumerate(root.to_ambient().to_vector()) if c).lstrip('+')
root_of = {str(r): r for r in rg4.roots()}

def in_eps(v):
    """v (quadratic in the currents e_alpha) in the notation of Perše: e_{α}(-1) e_{β}(-1) 1."""
    out = ''
    for key, c in sorted(v.monomial_coefficients().items()):
        assert all(n == 0 for _, n in key)        # no derivatives occur here
        e = ''.join('e_{%s}(-1)' % eps(root_of[D4.variable_names()[i]]) for i, _ in key)
        out += (' - ' if c < 0 else ' + ') + ('' if abs(c) == 1 else '%s ' % abs(c)) + e
    return '(' + out[3:] + ') 1' if out.startswith(' + ') else '(-' + out[3:] + ') 1'

sv4 = {}
for i in (1, 3, 4):
    src = D4.weight_space(2, horizontal_grade=two_omega(i))
    K = condition_matrix(D4, rg4, src, QQ).right_kernel_matrix()
    assert K.nrows() == 1
    sv4[i] = vectors_from(K.rows(), src)[0]
    print('v%d =' % i, sv4[i])
    print('    =', in_eps(sv4[i]))

# all of g^_+ kills them: x_(n) v = 0 for n >= 1 and all currents x, and x_(0) v = 0 for the positive root vectors
positive = [D4.gen(str(r)) for r in rg4.positive_roots()]
print('singular:', all(x.nproduct(v, n) == 0 for v in sv4.values() for x in gens4 for n in (1, 2))
      and all(x.nproduct(v, 0) == 0 for v in sv4.values() for x in positive))

v1 = :alpha[1] alpha[1] + 2*alpha[2] + alpha[3] + alpha[4]: + :alpha[1] + alpha[2] alpha[1] + alpha[2] + alpha[3] + alpha[4]: - :alpha[1] + alpha[2] + alpha[3] alpha[1] + alpha[2] + alpha[4]:
    = (e_{ε1-ε2}(-1)e_{ε1+ε2}(-1) + e_{ε1-ε3}(-1)e_{ε1+ε3}(-1) - e_{ε1-ε4}(-1)e_{ε1+ε4}(-1)) 1


v3 = :alpha[3] alpha[1] + 2*alpha[2] + alpha[3] + alpha[4]: - :alpha[2] + alpha[3] alpha[1] + alpha[2] + alpha[3] + alpha[4]: + :alpha[2] + alpha[3] + alpha[4] alpha[1] + alpha[2] + alpha[3]:
    = (e_{ε3-ε4}(-1)e_{ε1+ε2}(-1) - e_{ε2-ε4}(-1)e_{ε1+ε3}(-1) + e_{ε2+ε3}(-1)e_{ε1-ε4}(-1)) 1
v4 = :alpha[4] alpha[1] + 2*alpha[2] + alpha[3] + alpha[4]: - :alpha[2] + alpha[4] alpha[1] + alpha[2] + alpha[3] + alpha[4]: + :alpha[2] + alpha[3] + alpha[4] alpha[1] + alpha[2] + alpha[4]:
    = (e_{ε3+ε4}(-1)e_{ε1+ε2}(-1) - e_{ε2+ε4}(-1)e_{ε1+ε3}(-1) + e_{ε2+ε3}(-1)e_{ε1+ε4}(-1)) 1
singular: True


### Sums of ideals

Let $I_A = \sum_{i \in A} \langle v_i \rangle$ for the nonempty subsets $A \subseteq \{1, 3, 4\}$.

- At weight 2, $\langle v_i \rangle$ is the 35-dimensional $\mathfrak{so}_8$-module $L(2\omega_i)$ that $v_i$ generates, since the positive modes
  kill $v_i$.
- The three modules are inequivalent, so the sums are direct at weight 2, with dimensions 35, 70 and 105.
- `I <= J` checks that the generators of `I` lie in `J`. The table is the Boolean lattice of subsets of $\{1, 3, 4\}$.

In [17]:
# The ideals I_A = sum over i in A of <v_i>, for the nonempty subsets A of {1, 3, 4}
I4 = {(i,): D4.ideal([sv4[i]], names=['v%d' % i]) for i in (1, 3, 4)}
for A in Subsets([1, 3, 4]):
    if len(A) > 1:
        I4[tuple(A)] = sum(I4[(i,)] for i in A)       # I + J + ...
I4[(1, 3, 4)].variable_names()                    # the names of the summands are kept

('v1', 'v3', 'v4')

In [18]:
names4 = {A: '+'.join('I%d' % i for i in A) for A in I4}
print('dimension at weight 2 (of 434):', {names4[A]: J.weight_space(2).dimension() for A, J in I4.items()})
print()
print('containment I <= J (rows I, columns J):')
table([[''] + [names4[B] for B in I4]] + [[names4[A]] + [I4[A] <= I4[B] for B in I4] for A in I4])

dimension at weight 2 (of 434): {'I1': 35, 'I3': 35, 'I4': 35, 'I1+I3': 70, 'I1+I4': 70, 'I3+I4': 70, 'I1+I3+I4': 105}

containment I <= J (rows I, columns J):


,I1,I3,I4,I1+I3,I1+I4,I3+I4,I1+I3+I4
I1,\(\mathrm{True}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{True}\),\(\mathrm{True}\),\(\mathrm{False}\),\(\mathrm{True}\)
I3,\(\mathrm{False}\),\(\mathrm{True}\),\(\mathrm{False}\),\(\mathrm{True}\),\(\mathrm{False}\),\(\mathrm{True}\),\(\mathrm{True}\)
I4,\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{True}\),\(\mathrm{False}\),\(\mathrm{True}\),\(\mathrm{True}\),\(\mathrm{True}\)
I1+I3,\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{True}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{True}\)
I1+I4,\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{True}\),\(\mathrm{False}\),\(\mathrm{True}\)
I3+I4,\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{True}\),\(\mathrm{True}\)
I1+I3+I4,\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{False}\),\(\mathrm{True}\)


Membership in a sum, with a certificate. Also a single generator whose ideal is a sum: the zero modes of the Cartan
currents separate the components of $v_3 + v_4$, of different grades.

In [19]:
v1, v3, v4 = sv4[1], sv4[3], sv4[4]
f1 = D4.gen(str(-rg4.simple_root(1)))       # e_{-α1}
w = f1.nproduct(v1, 0) + v3                        # a lowered v1 plus v3
print('w in I1+I3:', w in I4[(1, 3)], '  in I1:', w in I4[(1,)], '  in I3:', w in I4[(3,)])
print('certificate:', I4[(1, 3)].express(w))

# one generator, but the ideal is a sum: the zero modes of the Cartan currents separate the components of v3 + v4
J34 = D4.ideal([v3 + v4])
print('<v3 + v4> == I3 + I4:', J34 <= I4[(3, 4)] and I4[(3, 4)] <= J34, '  <v3 + v4> <= I3:', J34 <= I4[(3,)])

w in I1+I3: True   in I1: False   in I3: False
certificate: v3 - alpha[1]_(0) v1


<v3 + v4> == I3 + I4: True   <v3 + v4> <= I3: False


### The simple quotient

By Perše's Thm 4.2, $I_1 + I_3 + I_4$ is the maximal ideal, so $V^{-2}(\mathfrak{so}_8)/(I_1 + I_3 + I_4) = L_{-2}(D_4)$.

Its graded dimensions agree with Table 5 of Beem, Lemos, Liendo, Peelaers, Rastelli and van Rees,
[arXiv:1312.5344](https://arxiv.org/abs/1312.5344). There $L_{-2}(D_4)$ is the chiral algebra of 4d $\mathcal N = 2$ $SU(2)$ SQCD with four flavours.

| level | $\mathfrak{so}_8$-modules | dimension |
|---|---|---|
| 2 | $1 + 28 + 300$ | 329 |
| 3 | $1 + 2 \cdot 28 + 300 + 350 + 1925$ | 2632 |

Weight 3 takes about 30 s, mostly the dense echelon form of the ideal's weight-3 space.

In [20]:
# The simple quotient L_{-2}(D4) (Perše, Thm 4.2) against Beem et al., Table 5: 1, 28, 1+28+300, 1+2*28+300+350+1925
L4 = D4 / I4[(1, 3, 4)]
print('V / (I1+I3+I4):', [L4.weight_space(n).dimension() for n in range(4)], ' expected:', [1, 28, 329, 2632])
print('V / I1 and V / (I1+I3) at weight 2:', (D4 / I4[(1,)]).weight_space(2).dimension(),
      (D4 / I4[(1, 3)]).weight_space(2).dimension())

V / (I1+I3+I4): [1, 28, 329, 2632]  expected: [1, 28, 329, 2632]
V / I1 and V / (I1+I3) at weight 2: 399 364
